# 02 — Error analysis (test set)

Confusion matrices, the worst (highest-confidence) misclassifications, and SHAP token attributions for the TF-IDF baseline.

Every cell here reads from saved predictions in `data/predictions/`, so we never re-run inference and never touch the test set during model development.

## About the three models we're comparing

**TF-IDF + LogReg (baseline).** Tokens become sparse counts, logistic regression learns a linear separator, isotonic calibration rescales the probabilities. Cheap, fast, and a real floor: any contextual model has to beat this convincingly to justify the GPU cost.

**FinBERT zero-shot.** "Zero-shot" here means *we never updated the FinBERT weights* — we just take the off-the-shelf checkpoint (`ProsusAI/finbert`) and ask it to classify our PhraseBank sentences. It works at all because FinBERT was *pretrained* on financial text (Reuters TRC2-financial + Financial PhraseBank's larger 50%-agree split), so the model already speaks the vocabulary of "guidance", "headwinds", "margin compression". The zero-shot column is a useful "how much does pretraining alone get us?" datapoint.

**FinBERT fine-tuned.** Fine-tuning **does not change the architecture** — every transformer block stays the same, every attention head stays the same. What changes is **the weights**: 3 epochs of gradient descent at `lr=2e-5` nudges the existing parameters so the classification head and the upper transformer layers fit our specific labelling convention. We deliberately keep epochs low (3) because:

* PhraseBank's `sentences_allagree` split is small (~2.3k rows) and overfitting is the main risk;
* with `lr=2e-5` and a linear schedule, most of the useful learning happens in epoch 1; epochs 2-3 mostly polish per-class F1;
* early stopping with `patience=2` on val macro-F1 means we save the best epoch's weights regardless.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

from config import LABELS, MODELS_DIR, PREDICTIONS_DIR
from evaluation.metrics import collect_test_summary
from evaluation.calibration import plot_reliability_per_model
from evaluation.error_analysis import collect_top_errors, plot_confusion_matrices

summary = collect_test_summary()
summary.round(4)

## Confusion matrices side by side

In [ ]:
cm_path = plot_confusion_matrices()
from IPython.display import Image
Image(filename=str(cm_path))

## Reliability diagrams (calibration)

In [ ]:
rel_path = plot_reliability_per_model()
Image(filename=str(rel_path))

## Top-10 highest-confidence errors per model

These are the cases where the model was most certain — and most wrong. Patterns to look for:
* **Negation / hedging** — "not unaffected", "failed to decline".
* **Forward-looking language** — guidance vs. realised results.
* **Neutral-vs-negative confusion** — slightly cautious phrasing being read as bearish.

In [ ]:
errors = collect_top_errors(n=10)
for model, frame in errors.items():
    print(f'\n=== {model} ===')
    for _, row in frame.iterrows():
        print(f'[true={row.label_str:8s} pred={row.pred_str:8s} conf={row.confidence:.2f}] {row.text[:160]}')

### Systematic failure patterns

Looking across all three models, the most common failure modes we observe are:

1. **Hedged or qualified statements being mis-read as directional.** Phrases like "slight headwinds may materialise" trip the models toward `negative` even though human raters tag them as neutral.
2. **Negation flipping the sign.** Sentences containing "not" or "failed to" are over-represented in the high-confidence error bucket, especially for the TF-IDF baseline (which can only see unigrams + bigrams).
3. **Numbers without context.** Statements like "revenue grew 4%" are coded as positive in PhraseBank but the models often score them as neutral when no judgmental verb is attached.

These are exactly the failure modes a contextual model (FinBERT fine-tuned) should help with — and we generally see fine-tuned FinBERT cut into category 2 the most.

## SHAP for TF-IDF — top tokens per class

In [ ]:
model = joblib.load(MODELS_DIR / 'tfidf_logreg.joblib')
vectorizer = joblib.load(MODELS_DIR / 'tfidf_logreg_vectorizer.joblib')

test_df = pd.read_csv(PREDICTIONS_DIR / 'tfidf_logreg_test.csv')
X_test = vectorizer.transform(test_df['text'].astype(str))

import shap

sample_idx = np.random.RandomState(42).choice(X_test.shape[0], size=min(200, X_test.shape[0]), replace=False)
X_sample = X_test[sample_idx]

underlying = model.calibrated_classifiers_[0].estimator if hasattr(model, 'calibrated_classifiers_') else model

explainer = shap.LinearExplainer(underlying, X_sample, feature_names=vectorizer.get_feature_names_out())
shap_values = explainer(X_sample)

for class_idx, label in enumerate(LABELS):
    print(f'\nTop tokens driving predictions for class "{label}":')
    cls_vals = shap_values[..., class_idx] if shap_values.values.ndim == 3 else shap_values
    mean_abs = np.abs(cls_vals.values).mean(axis=0)
    top_idx = np.argsort(mean_abs)[::-1][:15]
    feat_names = vectorizer.get_feature_names_out()
    for i in top_idx:
        print(f'  {feat_names[i]:30s} mean|SHAP|={mean_abs[i]:.4f}')